In [1]:
import random
import math
import numpy as np

In [2]:
BET = 5
NUMBER_OF_DECKS = 6
SURRENDER_ALLOWED = True
MAX_HANDS = 4          # table rule: 3 splits = 4 hands
DOUBLE_AFTER_SPLIT = True

bankroll = 1000
decks =  []
dealer_hand = []

lose_count = 0
win_count = 0
insurance_count = 0
double_count = 0


In [3]:
def new_hand(cards, bet_size):
    return {'cards': cards,
            'bet_size': bet_size,
            'surrendered': False,
            'done': False}

def reset():
    global decks, hands, running_count, true_count
    decks = NUMBER_OF_DECKS * 4 * [2,3,4,5,6,7,8,9,10,10,10,10,11]
    random.shuffle(decks)
    dealer_hand.clear()
    hands = []          # starts empty, grows as hands are split
    running_count = 0
    true_count = 0


def deal_dealer_hand():
    dealer_hand.append(decks.pop())
    dealer_hand.append(decks.pop())

def lose(HID):
    global lose_count, bankroll
    lose_count += 1
    bankroll -= hands[HID]['bet_size']
    return

def win(HID):
    global win_count, bankroll
    win_count += 1
    bankroll += hands[HID]['bet_size']
    return

def insurance ():
    global bankroll
    insurance_bet = BET/2
    bankroll -= insurance_bet
    if sum(dealer_hand) == 21:
        bankroll += 3 * insurance_bet
        lose(0)

def deal():
    hands.append(new_hand([decks.pop(), decks.pop()], BET))
    deal_dealer_hand()
    if dealer_hand[1] == 11 and true_count >= 3:
        insurance()

def split_logic(HID):
    if hands[HID]['cards'][0] in [11,8]:
        return True
    elif hands[HID]['cards'][0] == 9 and dealer_hand[1] not in [7,10,11]:
        return True
    elif hands[HID]['cards'][0] == 7 and dealer_hand[1] <= 7:
        return True
    elif hands[HID]['cards'][0] == 6 and dealer_hand[1] <= 6 and (dealer_hand[1] > 2 or DOUBLE_AFTER_SPLIT):
        return True
    elif hands[HID]['cards'][0] == 4 and dealer_hand[1] in [5,6] and DOUBLE_AFTER_SPLIT:
        return True
    elif hands[HID]['cards'][0] in [3,2] and dealer_hand[1] <= 6 and (dealer_hand[1] > 3 or DOUBLE_AFTER_SPLIT):
        return True
    else: 
        False


def split(HID):
    if len(hands) >= MAX_HANDS and split_logic(HID):
        return False
    hand = hands[HID]
    moved = hand['cards'].pop()             # take one card off the original
    hand['cards'].append(decks.pop())       # redraw to the original
    hands.append(new_hand([moved, decks.pop()], hand['bet_size']))
    return True

def surrender(HID):
    global bankroll
    hand = hands[HID]
    bankroll -= hand['bet_size'] / 2
    hand['surrendered'] = True
    hand['done'] = True

def player_logic(HID):
    hand = hands[HID]
    total = sum(hand['cards'])
    upcard = dealer_hand[1]
    if SURRENDER_ALLOWED and ((total == 16 and upcard in [9,10,11]) or (total == 15 and upcard == 10)):
        surrender(HID)

def play():
    HID = 0
    while HID < len(hands):     # len() re-read each pass, so split hands get played
        player_logic(HID)
        HID += 1


In [4]:
def main():
    reset()
    deal()
    play()
    print(hands,  dealer_hand)

if __name__ == "__main__":
    main()


[{'cards': [10, 3], 'bet_size': 5, 'surrendered': False, 'done': False}] [10, 4]
